<a href="https://colab.research.google.com/github/chawlaniharika23/Fraud_Detection_Research/blob/main/notebooks/06_improved_final_evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 06 — Final Test Evaluation and PaySim Comparison

## Financial Fraud Detection: IEEE-CIS and PaySim

### Objective
Evaluate the frozen IEEE-CIS base classifiers and logistic-stacking ensemble on the untouched chronological test partition. Then train and evaluate PaySim models using the chronological split artifacts generated by Notebook 02.

### Methodological safeguards
- IEEE-CIS test data is loaded only in this notebook, after model and threshold selection.
- The stacking threshold comes from Notebook 05's later validation segment and is not changed here.
- IEEE-CIS base models use a common 0.50 threshold for a transparent reference comparison; the stack uses its validation-selected threshold.
- PaySim uses the exact chronological split files saved by Notebook 02. No new random split is created.
- PaySim imputers/scalers are fitted on training data only; validation selects thresholds; test is evaluated once.
- PaySim real-time and extended feature sets are reported separately. Extended features include post-transaction balance information and represent an offline setting, not a strict real-time scoring scenario.

**Terminology:** the IEEE-CIS ensemble is static probability-level logistic stacking. The legacy output label `Dynamic_Stacking` is retained only where needed for compatibility with Notebook 07.

## 1. Imports, reproducibility and paths

In [ ]:
import os
import gc
import json
import time
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import scipy
import sklearn

from scipy.sparse import load_npz
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, average_precision_score, matthews_corrcoef,
    confusion_matrix, classification_report, brier_score_loss,
    precision_recall_curve
)
from IPython.display import display

warnings.filterwarnings("ignore", category=FutureWarning)
RANDOM_STATE = 42
BASE_DIR = Path("/content/drive/MyDrive/fraud-detection-research")
PROCESSED_DIR = BASE_DIR / "data" / "processed"
RECOVERY_DIR = PROCESSED_DIR / "recovered"
RESULTS_DIR = BASE_DIR / "results"
BASE_MODEL_DIR = RESULTS_DIR / "base_models"
STACKING_DIR = RESULTS_DIR / "dynamic_stacking"
IEEE_EVAL_DIR = RESULTS_DIR / "ieee_evaluation"
PAYSIM_EVAL_DIR = RESULTS_DIR / "paysim_evaluation"

for folder in [IEEE_EVAL_DIR, PAYSIM_EVAL_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("SciPy:", scipy.__version__)
print("scikit-learn:", sklearn.__version__)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 2. Evaluation helper

In [ ]:
def evaluate_binary_model(y_true, probabilities, threshold=0.5):
    y_true = np.asarray(y_true, dtype=np.int8)
    probabilities = np.asarray(probabilities, dtype=np.float64).reshape(-1)

    if len(y_true) != len(probabilities):
        raise ValueError("Target and probability lengths do not match.")
    if not np.isfinite(probabilities).all():
        raise ValueError("Predictions contain NaN or infinity.")
    if len(np.unique(y_true)) < 2:
        raise ValueError("Evaluation partition must contain both classes.")

    probabilities = np.clip(probabilities, 0.0, 1.0)
    predictions = (probabilities >= threshold).astype(np.int8)
    tn, fp, fn, tp = confusion_matrix(
        y_true, predictions, labels=[0, 1]
    ).ravel()

    return {
        "Threshold": float(threshold),
        "Accuracy": accuracy_score(y_true, predictions),
        "Balanced_Accuracy": balanced_accuracy_score(y_true, predictions),
        "Precision": precision_score(y_true, predictions, zero_division=0),
        "Recall": recall_score(y_true, predictions, zero_division=0),
        "F1": f1_score(y_true, predictions, zero_division=0),
        "ROC_AUC": roc_auc_score(y_true, probabilities),
        "PR_AUC": average_precision_score(y_true, probabilities),
        "MCC": matthews_corrcoef(y_true, predictions),
        "FPR": fp / (fp + tn) if (fp + tn) else 0.0,
        "Brier_Score": brier_score_loss(y_true, probabilities),
        "TN": int(tn), "FP": int(fp), "FN": int(fn), "TP": int(tp)
    }


def choose_f1_threshold(y_true, probabilities):
    precision, recall, thresholds = precision_recall_curve(
        y_true, probabilities
    )
    if len(thresholds) == 0:
        raise ValueError("No threshold candidates available.")
    f1_values = (
        2 * precision[:-1] * recall[:-1]
        / (precision[:-1] + recall[:-1] + 1e-12)
    )
    best_idx = int(np.nanargmax(f1_values))
    return float(thresholds[best_idx])

## 3. IEEE-CIS: load the untouched test partition

This is the first and only use of the IEEE-CIS test labels for model evaluation. Do not tune models, feature selection, or thresholds based on the results in this notebook.

In [ ]:
IEEE_TEST_MATRIX = RECOVERY_DIR / "ieee_X_test_selected_100.npz"
IEEE_TEST_TARGET = PROCESSED_DIR / "ieee_y_test.parquet"

required_ieee = [
    IEEE_TEST_MATRIX, IEEE_TEST_TARGET,
    STACKING_DIR / "meta_model.joblib",
    STACKING_DIR / "stacking_config.joblib"
]
missing = [str(path) for path in required_ieee if not path.exists()]
if missing:
    raise FileNotFoundError(
        "Missing required IEEE-CIS artifacts:\n" + "\n".join(missing)
        + "\nRun Notebooks 03, 04 and 05 successfully first."
    )

X_ieee_test = load_npz(IEEE_TEST_MATRIX).tocsr().astype(np.float32)
y_ieee_test = pd.read_parquet(IEEE_TEST_TARGET)["isFraud"].to_numpy(dtype=np.int8)

if X_ieee_test.shape[0] != len(y_ieee_test):
    raise ValueError("IEEE-CIS test matrix and labels are misaligned.")
if X_ieee_test.shape[1] != 100:
    raise ValueError(f"Expected 100 selected features; found {X_ieee_test.shape[1]}.")

print("IEEE-CIS test matrix:", X_ieee_test.shape)
print("Test fraud cases:", int(y_ieee_test.sum()))
print("Test fraud rate:", f"{y_ieee_test.mean()*100:.4f}%")

## 4. IEEE-CIS: load frozen models and stacking configuration

In [ ]:
IEEE_MODEL_NAMES = ["LR", "DT", "RF", "GBT", "SVM", "MLP"]
missing_models = [
    str(BASE_MODEL_DIR / f"{name}_model.joblib")
    for name in IEEE_MODEL_NAMES
    if not (BASE_MODEL_DIR / f"{name}_model.joblib").exists()
]
if missing_models:
    raise FileNotFoundError("Missing base models:\n" + "\n".join(missing_models))

ieee_models = {
    name: joblib.load(BASE_MODEL_DIR / f"{name}_model.joblib")
    for name in IEEE_MODEL_NAMES
}
ieee_meta_model = joblib.load(STACKING_DIR / "meta_model.joblib")
ieee_stacking_config = joblib.load(STACKING_DIR / "stacking_config.joblib")

stacking_model_order = ieee_stacking_config["model_names"]
ieee_stacking_threshold = float(ieee_stacking_config["threshold"])

if stacking_model_order != IEEE_MODEL_NAMES:
    raise ValueError(
        "Model order mismatch between Notebook 04/05 and stacking configuration."
    )
if not 0.0 <= ieee_stacking_threshold <= 1.0:
    raise ValueError("Saved stacking threshold is outside [0, 1].")

print("Loaded base models:", list(ieee_models))
print("Stacking model order:", stacking_model_order)
print("Frozen validation threshold:", ieee_stacking_threshold)

## 5. IEEE-CIS: generate test probabilities and evaluate

In [ ]:
ieee_test_probabilities = {}

for name, model in ieee_models.items():
    probability = model.predict_proba(X_ieee_test)[:, 1]
    if not np.isfinite(probability).all():
        raise ValueError(f"{name} returned non-finite test probabilities.")
    ieee_test_probabilities[name] = probability.astype(np.float32)
    print(f"{name}: test predictions generated.")

stack_meta_features = np.column_stack([
    ieee_test_probabilities[name] for name in stacking_model_order
])
stack_probability = ieee_meta_model.predict_proba(stack_meta_features)[:, 1]
ieee_test_probabilities["Dynamic_Stacking"] = stack_probability.astype(np.float32)

ieee_result_rows = []
for model_name, probabilities in ieee_test_probabilities.items():
    threshold = (
        ieee_stacking_threshold
        if model_name == "Dynamic_Stacking"
        else 0.50
    )
    row = evaluate_binary_model(
        y_ieee_test, probabilities, threshold=threshold
    )
    row["Model"] = model_name
    ieee_result_rows.append(row)

ieee_results = pd.DataFrame(ieee_result_rows).sort_values(
    "PR_AUC", ascending=False
).reset_index(drop=True)

ieee_results.to_csv(IEEE_EVAL_DIR / "ieee_test_results.csv", index=False)
display(ieee_results.round(5))

## 6. IEEE-CIS: confusion matrix and classification report

The stacking operating threshold is the threshold selected in Notebook 05. The test labels are used only to calculate final metrics, not to revise this threshold.

In [ ]:
stack_test_labels = (
    stack_probability >= ieee_stacking_threshold
).astype(np.int8)

stack_cm = confusion_matrix(
    y_ieee_test, stack_test_labels, labels=[0, 1]
)
print("Static logistic stacking — IEEE-CIS test set")
print("Validation-selected threshold:", ieee_stacking_threshold)
print("\nConfusion matrix [ [TN, FP], [FN, TP] ]:")
print(stack_cm)
print("\nClassification report:")
print(classification_report(
    y_ieee_test,
    stack_test_labels,
    target_names=["Legitimate", "Fraud"],
    zero_division=0
))

## 7. IEEE-CIS: calculate improvement over the strongest base model

The strongest individual base classifier is identified by test PR-AUC for descriptive comparison. Because this selection uses test outcomes, treat the comparison as exploratory reporting rather than an unbiased model-selection procedure. The proposed ensemble itself was fixed before test evaluation.

Percentage improvement is reported only where the baseline metric is non-zero.

In [ ]:
base_only = ieee_results[ieee_results["Model"] != "Dynamic_Stacking"]
best_base = base_only.sort_values("PR_AUC", ascending=False).iloc[0]
stack_row = ieee_results[ieee_results["Model"] == "Dynamic_Stacking"].iloc[0]

improvement_rows = []
for metric in ["PR_AUC", "ROC_AUC", "F1", "Recall", "Precision", "MCC"]:
    baseline_value = float(best_base[metric])
    proposed_value = float(stack_row[metric])
    pct = (
        (proposed_value - baseline_value) / abs(baseline_value) * 100
        if baseline_value != 0 else np.nan
    )
    improvement_rows.append({
        "Metric": metric,
        "Best_Base_Model": best_base["Model"],
        "Best_Base_Value": baseline_value,
        "Stacking_Value": proposed_value,
        "Absolute_Difference": proposed_value - baseline_value,
        "Relative_Change_Percent": pct
    })

improvement_df = pd.DataFrame(improvement_rows)
improvement_df.to_csv(
    IEEE_EVAL_DIR / "ieee_stacking_improvement_vs_best_base.csv",
    index=False
)
print("Best individual model by test PR-AUC:", best_base["Model"])
display(improvement_df.round(6))

## 8. IEEE-CIS: save final evaluation summary

The summary records the actual method as static logistic stacking. The legacy result column name is retained so the existing cross-dataset summary notebook can still read the CSV.

In [ ]:
summary_text = f"""IEEE-CIS FINAL CHRONOLOGICAL TEST EVALUATION

Ensemble method: Static probability-level logistic stacking
Base classifiers: {', '.join(IEEE_MODEL_NAMES)}
Validation-selected threshold: {ieee_stacking_threshold:.8f}
Test rows: {len(y_ieee_test)}
Test fraud cases: {int(y_ieee_test.sum())}
Test fraud prevalence: {y_ieee_test.mean()*100:.6f}%

Stacking test metrics:
Precision: {stack_row['Precision']:.6f}
Recall: {stack_row['Recall']:.6f}
F1: {stack_row['F1']:.6f}
PR-AUC: {stack_row['PR_AUC']:.6f}
ROC-AUC: {stack_row['ROC_AUC']:.6f}
MCC: {stack_row['MCC']:.6f}
False-positive rate: {stack_row['FPR']:.6f}

Best individual base model by test PR-AUC (descriptive): {best_base['Model']}
Its test PR-AUC: {best_base['PR_AUC']:.6f}

The test set was not used for model fitting or threshold selection.
The best-base identification above is descriptive and must not be used to
retroactively tune the proposed model.
"""
(IEEE_EVAL_DIR / "ieee_research_interpretation.txt").write_text(summary_text)
print(summary_text)

## 9. PaySim: load the exact Notebook 02 chronological split files

Do not recreate the split from the full PaySim CSV. Notebook 02 already saved aligned real-time/extended feature matrices, labels, and step metadata for train, validation, and test.

In [ ]:
PAYSIM_SPLIT_FILES = {}
for split_name in ["train", "validation", "test"]:
    paths = {
        "realtime": PROCESSED_DIR / f"paysim_X_{split_name}_realtime.parquet",
        "extended": PROCESSED_DIR / f"paysim_X_{split_name}_extended.parquet",
        "target": PROCESSED_DIR / f"paysim_y_{split_name}.parquet",
        "step": PROCESSED_DIR / f"paysim_step_{split_name}.parquet"
    }
    missing = [str(path) for path in paths.values() if not path.exists()]
    if missing:
        raise FileNotFoundError(
            "Missing PaySim chronological split outputs from Notebook 02:\n"
            + "\n".join(missing)
        )
    PAYSIM_SPLIT_FILES[split_name] = paths

print("Found all PaySim chronological split files.")

## 10. PaySim: train models, select thresholds on validation, evaluate test

For each feature set, fit imputation and scaling using training data only. Fit Logistic Regression and Random Forest on the training partition, select each model's threshold using validation predictions, and evaluate the test partition once.

The real-time and extended results are separate experiments. The extended feature set contains post-transaction balance fields and must not be presented as a strict real-time fraud-scoring result.

In [ ]:
from sklearn.base import clone
from sklearn.utils.class_weight import compute_sample_weight

paysim_model_templates = {
    "LR": LogisticRegression(
        class_weight="balanced",
        max_iter=500,
        solver="lbfgs",
        random_state=RANDOM_STATE
    ),
    "RF": RandomForestClassifier(
        n_estimators=80,
        max_depth=12,
        min_samples_leaf=5,
        max_features="sqrt",
        class_weight="balanced_subsample",
        n_jobs=-1,
        random_state=RANDOM_STATE
    )
}

paysim_results_rows = []
paysim_threshold_records = []

for feature_set in ["realtime", "extended"]:
    print(f"\n===== PaySim {feature_set.upper()} feature set =====")

    # Load one split at a time for predictable memory usage.
    X_train_df = pd.read_parquet(PAYSIM_SPLIT_FILES["train"][feature_set])
    X_val_df = pd.read_parquet(PAYSIM_SPLIT_FILES["validation"][feature_set])
    X_test_df = pd.read_parquet(PAYSIM_SPLIT_FILES["test"][feature_set])

    y_train_ps = pd.read_parquet(PAYSIM_SPLIT_FILES["train"]["target"])["isFraud"].to_numpy(dtype=np.int8)
    y_val_ps = pd.read_parquet(PAYSIM_SPLIT_FILES["validation"]["target"])["isFraud"].to_numpy(dtype=np.int8)
    y_test_ps = pd.read_parquet(PAYSIM_SPLIT_FILES["test"]["target"])["isFraud"].to_numpy(dtype=np.int8)

    if not (list(X_train_df.columns) == list(X_val_df.columns) == list(X_test_df.columns)):
        raise ValueError(f"PaySim {feature_set} feature columns are not aligned.")
    if not (len(X_train_df) == len(y_train_ps) and len(X_val_df) == len(y_val_ps)
            and len(X_test_df) == len(y_test_ps)):
        raise ValueError(f"PaySim {feature_set} feature/label alignment error.")

    # Fit imputation/scaling on training data only.
    imputer = SimpleImputer(strategy="median")
    scaler = StandardScaler()

    Xtr = imputer.fit_transform(X_train_df).astype(np.float32)
    Xva = imputer.transform(X_val_df).astype(np.float32)
    Xte = imputer.transform(X_test_df).astype(np.float32)

    Xtr = scaler.fit_transform(Xtr).astype(np.float32)
    Xva = scaler.transform(Xva).astype(np.float32)
    Xte = scaler.transform(Xte).astype(np.float32)

    print("Rows:", len(y_train_ps), len(y_val_ps), len(y_test_ps))
    print("Fraud rates (%):",
          round(y_train_ps.mean()*100, 5),
          round(y_val_ps.mean()*100, 5),
          round(y_test_ps.mean()*100, 5))

    for model_name, template in paysim_model_templates.items():
        model = clone(template)
        fit_start = time.time()
        model.fit(Xtr, y_train_ps)
        fit_seconds = time.time() - fit_start

        val_probability = model.predict_proba(Xva)[:, 1]
        selected_threshold = choose_f1_threshold(y_val_ps, val_probability)

        test_start = time.time()
        test_probability = model.predict_proba(Xte)[:, 1]
        predict_seconds = time.time() - test_start

        metrics = evaluate_binary_model(
            y_test_ps, test_probability, threshold=selected_threshold
        )
        metrics.update({
            "Dataset": "PaySim",
            "Feature_Set": feature_set.capitalize(),
            "Model": model_name,
            "Training_Time_Seconds": fit_seconds,
            "Test_Prediction_Time_Seconds": predict_seconds
        })
        paysim_results_rows.append(metrics)
        paysim_threshold_records.append({
            "Feature_Set": feature_set.capitalize(),
            "Model": model_name,
            "Validation_Selected_Threshold": selected_threshold
        })

        joblib.dump(
            {"model": model, "imputer": imputer, "scaler": scaler,
             "feature_columns": list(X_train_df.columns),
             "threshold": selected_threshold},
            PAYSIM_EVAL_DIR / f"{feature_set}_{model_name}_pipeline.joblib",
            compress=3
        )

        print(
            f"{model_name}: PR-AUC={metrics['PR_AUC']:.6f}, "
            f"F1={metrics['F1']:.6f}, threshold={selected_threshold:.6f}"
        )

        del model, val_probability, test_probability
        gc.collect()

    del X_train_df, X_val_df, X_test_df, Xtr, Xva, Xte
    del y_train_ps, y_val_ps, y_test_ps, imputer, scaler
    gc.collect()

paysim_results = pd.DataFrame(paysim_results_rows).sort_values(
    "PR_AUC", ascending=False
).reset_index(drop=True)
paysim_thresholds = pd.DataFrame(paysim_threshold_records)

paysim_results.to_csv(PAYSIM_EVAL_DIR / "paysim_results.csv", index=False)
paysim_thresholds.to_csv(PAYSIM_EVAL_DIR / "paysim_thresholds.csv", index=False)

display(paysim_results.round(6))
display(paysim_thresholds.round(6))

## 11. PaySim: result interpretation and feature-set caveat

PaySim real-time and extended scores must be interpreted separately. A better extended score may result from access to post-transaction balance fields that are unavailable at the intended real-time scoring point.

In [ ]:
best_paysim = paysim_results.sort_values("PR_AUC", ascending=False).iloc[0]
best_realtime = paysim_results[paysim_results["Feature_Set"] == "Realtime"].sort_values(
    "PR_AUC", ascending=False
).iloc[0]

paysim_summary = f"""PAYSIM CHRONOLOGICAL TEST EVALUATION

Best overall configuration by test PR-AUC (descriptive):
Feature set: {best_paysim['Feature_Set']}
Model: {best_paysim['Model']}
PR-AUC: {best_paysim['PR_AUC']:.6f}
F1: {best_paysim['F1']:.6f}
Precision: {best_paysim['Precision']:.6f}
Recall: {best_paysim['Recall']:.6f}

Best real-time feature configuration:
Model: {best_realtime['Model']}
PR-AUC: {best_realtime['PR_AUC']:.6f}
F1: {best_realtime['F1']:.6f}

The extended feature set includes post-transaction balance information and
should be interpreted as an offline experiment, not a strict real-time result.
Thresholds were selected on validation data before test evaluation.
"""
(PAYSIM_EVAL_DIR / "paysim_research_interpretation.txt").write_text(paysim_summary)
print(paysim_summary)

## 12. Combined result exports and plots

The datasets have different feature spaces, fraud prevalence, and data-generation processes. Compare models within each dataset; do not interpret raw PR-AUC values across datasets as a direct ranking.

In [ ]:
import matplotlib.pyplot as plt

# Keep the expected filenames and columns used by Notebook 07.
ieee_export = ieee_results.copy()
ieee_export["Dataset"] = "IEEE-CIS"
paysim_export = paysim_results.copy()
paysim_export["Dataset"] = "PaySim"

combined_results = pd.concat(
    [ieee_export, paysim_export],
    ignore_index=True,
    sort=False
)
combined_results.to_csv(
    RESULTS_DIR / "all_test_experiment_results.csv",
    index=False
)

plt.figure(figsize=(9, 5))
plt.bar(ieee_results["Model"], ieee_results["PR_AUC"])
plt.xlabel("IEEE-CIS model")
plt.ylabel("Test PR-AUC")
plt.xticks(rotation=25)
plt.tight_layout()
plt.savefig(IEEE_EVAL_DIR / "ieee_test_pr_auc_comparison.png", dpi=300)
plt.show()

plt.figure(figsize=(9, 5))
paysim_labels = paysim_results["Feature_Set"] + " " + paysim_results["Model"]
plt.bar(paysim_labels, paysim_results["PR_AUC"])
plt.xlabel("PaySim feature set and model")
plt.ylabel("Test PR-AUC")
plt.xticks(rotation=25)
plt.tight_layout()
plt.savefig(PAYSIM_EVAL_DIR / "paysim_test_pr_auc_comparison.png", dpi=300)
plt.show()

print("Combined results and comparison plots saved.")

## 13. Final artifact validation and research handoff

In [ ]:
required_outputs = [
    IEEE_EVAL_DIR / "ieee_test_results.csv",
    IEEE_EVAL_DIR / "ieee_research_interpretation.txt",
    IEEE_EVAL_DIR / "ieee_stacking_improvement_vs_best_base.csv",
    PAYSIM_EVAL_DIR / "paysim_results.csv",
    PAYSIM_EVAL_DIR / "paysim_thresholds.csv",
    PAYSIM_EVAL_DIR / "paysim_research_interpretation.txt",
    IEEE_EVAL_DIR / "ieee_test_pr_auc_comparison.png",
    PAYSIM_EVAL_DIR / "paysim_test_pr_auc_comparison.png"
]
missing_outputs = [str(path) for path in required_outputs if not path.exists()]
if missing_outputs:
    raise FileNotFoundError("Missing final outputs:\n" + "\n".join(missing_outputs))

print("Notebook 06 completed.")
print("\nIEEE-CIS results:")
display(ieee_results.round(5))
print("\nPaySim results:")
display(paysim_results.round(5))
print("\nNext: Notebook 07 — cross-dataset result synthesis.")